# LangGraph 마스터하기

> Chain은 직선도로, Graph는 도시 전체 도로망이다 — 현실의 문제는 직선으로 풀리지 않는다

LangChain의 Chain은 A→B→C 직선 파이프라인이다. 그러나 현실의 에이전트는 **분기하고, 실패하면 재시도하고, 사람의 승인을 기다린다.**  
LangGraph는 이 복잡성을 **그래프 구조(StateGraph)**로 정면 돌파한다.

### 학습 목표

| # | 목표 | 핵심 |
|---|------|------|
| 1 | **StateGraph** 구조 이해 | 상태를 공유하는 노드와 엣지의 그래프 |
| 2 | **Nodes / Edges / Conditional Edges** 설계 | 처리 단계와 분기 로직 |
| 3 | **Human-in-the-Loop** 패턴 적용 | 위험 작업 전 사람 승인 |
| 4 | **멀티스텝 에이전트** 구현 | 실패→재시도→성공 루프 |
| 5 | **체크포인팅** 이해 | 상태 저장/복원으로 대화 이력 관리 |

In [ ]:
# !pip install langgraph langchain langchain-ollama -q
# 이 노트북은 LangGraph 개념을 순수 Python으로 시뮬레이션합니다.
# 실제 LangGraph 설치 없이도 핵심 개념을 학습할 수 있습니다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from typing import TypedDict, Optional, List, Dict, Any
from dataclasses import dataclass, field
import random
import time

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("Phase 10-2: LangGraph 마스터하기")
print("=" * 50)
print("핵심: Chain은 직선도로, Graph는 도시 전체 도로망이다.")
print("현실의 문제는 분기와 루프로 이루어져 있다.")

---
## 1. LangChain vs LangGraph — 왜 Graph가 필요한가

| 비교 항목 | LangChain (LCEL) | LangGraph |
|----------|-----------------|----------|
| **구조** | 직선형 체인 (A→B→C) | 그래프 (분기 + 루프) |
| **조건 분기** | RunnableBranch (제한적) | Conditional Edge (자유롭게) |
| **루프/재시도** | 직접 구현 필요 | 내장 지원 |
| **상태 관리** | 수동 (변수 전달) | StateGraph (자동) |
| **Human-in-the-Loop** | 미지원 | 내장 interrupt 지원 |
| **체크포인팅** | 미지원 | 내장 (중단/재개) |
| **비유** | 직선도로 | 도시 전체 도로망 |

Chain은 `질문 → 프롬프트 → LLM → 파서` 의 직선이다.  
Graph는 `질문 → 분류 → (SQL경로 | 일반경로) → 실행 → (성공 | 실패→재시도)` 의 **도로망**이다.

In [ ]:
# === Chain vs Graph 시각화 ===

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 좌측: Linear Chain (LangChain) ---
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('LangChain — 직선 체인', fontsize=14, fontweight='bold')

chain_nodes = [
    (5, 9, '질문 입력'),
    (5, 7, '프롬프트 구성'),
    (5, 5, 'LLM 호출'),
    (5, 3, '출력 파서'),
    (5, 1, '응답 반환'),
]

for x, y, label in chain_nodes:
    box = mpatches.FancyBboxPatch((x-1.2, y-0.4), 2.4, 0.8,
                                  boxstyle="round,pad=0.1", 
                                  facecolor='#339af0', edgecolor='black', linewidth=2)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center', fontsize=10, color='white', fontweight='bold')

for i in range(len(chain_nodes)-1):
    ax.annotate('', xy=(5, chain_nodes[i+1][1]+0.5), xytext=(5, chain_nodes[i][1]-0.5),
                arrowprops=dict(arrowstyle='->', lw=2, color='#333'))

ax.text(5, -0.3, '한 방향으로만 진행. 실패해도 재시도 불가.', 
        ha='center', fontsize=9, color='red', style='italic')

# --- 우측: Graph (LangGraph) ---
ax = axes[1]
ax.set_xlim(0, 12)
ax.set_ylim(-1, 11)
ax.axis('off')
ax.set_title('LangGraph — 그래프 (분기 + 루프)', fontsize=14, fontweight='bold')

graph_nodes = {
    'start':   (6, 10, '질문 분석', '#339af0'),
    'branch':  (6, 8, '조건 분기', '#ffa94d'),
    'sql':     (3, 6, 'SQL 생성', '#ff6b6b'),
    'general': (9, 6, '직접 응답', '#51cf66'),
    'exec':    (3, 4, 'SQL 실행', '#ff6b6b'),
    'check':   (3, 2, '성공?', '#ffa94d'),
    'result':  (6, 0, '최종 응답', '#51cf66'),
}

for key, (x, y, label, color) in graph_nodes.items():
    if '?' in label:
        diamond = plt.Polygon([(x, y+0.5), (x+0.8, y), (x, y-0.5), (x-0.8, y)], 
                              facecolor=color, edgecolor='black', linewidth=2)
        ax.add_patch(diamond)
        ax.text(x, y, label, ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    else:
        box = mpatches.FancyBboxPatch((x-1.0, y-0.35), 2.0, 0.7,
                                      boxstyle="round,pad=0.1",
                                      facecolor=color, edgecolor='black', linewidth=2)
        ax.add_patch(box)
        ax.text(x, y, label, ha='center', va='center', fontsize=9, fontweight='bold', color='white')

# Edges
edges = [
    ('start', 'branch', ''),
    ('branch', 'sql', 'SQL 필요'),
    ('branch', 'general', '일반 질문'),
    ('sql', 'exec', ''),
    ('exec', 'check', ''),
    ('general', 'result', ''),
]

edge_coords = [
    ((6, 9.6), (6, 8.4)),
    ((5.2, 7.7), (3.8, 6.4)),
    ((6.8, 7.7), (8.2, 6.4)),
    ((3, 5.6), (3, 4.4)),
    ((3, 3.6), (3, 2.5)),
    ((9, 5.6), (6.8, 0.4)),
]

for (start_xy, end_xy), (_, _, _, label, *_) in zip(edge_coords, [(0,0,0,''), (0,0,0,'SQL 필요'), (0,0,0,'일반 질문'), (0,0,0,''), (0,0,0,''), (0,0,0,'')]):
    ax.annotate('', xy=end_xy, xytext=start_xy,
                arrowprops=dict(arrowstyle='->', lw=2, color='#333'))

# Labels on edges
ax.text(4.2, 7.5, 'SQL 필요', fontsize=8, color='#666')
ax.text(7.2, 7.5, '일반 질문', fontsize=8, color='#666')

# Success edge
ax.annotate('', xy=(5.2, 0.3), xytext=(3.5, 1.6),
            arrowprops=dict(arrowstyle='->', lw=2, color='green'))
ax.text(3.8, 0.8, '성공', fontsize=8, color='green', fontweight='bold')

# Retry loop (failure → back to SQL generation)
ax.annotate('', xy=(1.5, 5.8), xytext=(1.5, 2.2),
            arrowprops=dict(arrowstyle='->', lw=2.5, color='red', 
                           connectionstyle='arc3,rad=-0.3'))
ax.text(0.2, 4, '실패\n재시도', fontsize=8, color='red', fontweight='bold', ha='center')

ax.text(6, -0.8, '분기, 루프, 조건부 경로. 실패하면 자동 재시도.', 
        ha='center', fontsize=9, color='green', style='italic')

plt.suptitle('Chain은 직선도로, Graph는 도시 전체 도로망', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n핵심 차이:")
print("  Chain: 질문→프롬프트→LLM→파서 (실패하면? 끝.)")
print("  Graph: 질문→분류→(경로A|경로B)→실행→(성공|실패→재시도) (실패해도 루프)")

---
## 2. LangGraph 핵심 개념

LangGraph의 4가지 핵심 구성요소:

| 개념 | 설명 | 비유 |
|------|------|------|
| **StateGraph** | 상태를 공유하는 그래프 정의 | 도시의 도로망 설계도 |
| **Nodes** | 각 처리 단계 (함수) | 도로 위의 교차로 |
| **Edges** | 노드 간 고정 연결 | 일방통행 도로 |
| **Conditional Edges** | 조건에 따른 분기 | 신호등이 있는 교차로 |

```
StateGraph(상태타입)
  ├── add_node("이름", 함수)          # 처리 단계 등록
  ├── add_edge("A", "B")             # A 끝나면 반드시 B로
  ├── add_conditional_edges("A", fn)  # A 끝나면 fn 결과에 따라 분기
  ├── set_entry_point("시작노드")      # 그래프 입구
  └── compile()                       # 실행 가능한 앱으로 컴파일
```

In [ ]:
# === LangGraph 코드 구조 상세 출력 ===

print("=" * 70)
print("LangGraph 코드 구조 — 5단계 패턴")
print("=" * 70)

print("""
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1단계: 상태(State) 정의 — TypedDict로 공유 상태 선언
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from typing import TypedDict
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    question: str           # 사용자 질문
    category: str           # 질문 분류 (sql / general / code)
    sql: str                # 생성된 SQL
    result: str             # 실행 결과
    error: str              # 에러 메시지
    retry_count: int        # 재시도 횟수
    final_answer: str       # 최종 응답
""")

print("""
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2단계: 노드 함수 정의 — 각 처리 단계를 함수로 작성
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def analyze_question(state: AgentState) -> dict:
    \"\"\"질문을 분석하여 카테고리 분류\"\"\"
    question = state['question']
    # LLM으로 질문 분류
    category = llm.invoke(f"분류: {question}")
    return {"category": category}

def generate_sql(state: AgentState) -> dict:
    \"\"\"SQL 생성\"\"\"
    sql = llm.invoke(f"SQL 생성: {state['question']}")
    return {"sql": sql, "retry_count": state.get('retry_count', 0) + 1}

def execute_sql(state: AgentState) -> dict:
    \"\"\"SQL 실행\"\"\"
    try:
        result = db.execute(state['sql'])
        return {"result": str(result), "error": ""}
    except Exception as e:
        return {"error": str(e)}
""")

print("""
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3단계: 조건 분기 함수 — 어디로 갈지 결정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def route_question(state: AgentState) -> str:
    \"\"\"질문 카테고리에 따라 경로 결정\"\"\"
    if state['category'] == 'sql':
        return 'generate_sql'     # SQL 경로
    return 'direct_answer'        # 일반 응답 경로

def should_retry(state: AgentState) -> str:
    \"\"\"실행 결과에 따라 재시도 여부 결정\"\"\"
    if not state.get('error'):
        return 'format_response'  # 성공 → 응답 생성
    if state['retry_count'] < 3:
        return 'generate_sql'     # 실패 → 재시도
    return 'fail_response'        # 3회 초과 → 실패 응답
""")

print("""
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4단계: 그래프 구성 — 노드와 엣지 연결
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

graph = StateGraph(AgentState)

# 노드 등록
graph.add_node("analyze", analyze_question)
graph.add_node("generate_sql", generate_sql)
graph.add_node("execute_sql", execute_sql)
graph.add_node("format_response", format_response)
graph.add_node("direct_answer", direct_answer)
graph.add_node("fail_response", fail_response)

# 시작점
graph.set_entry_point("analyze")

# 고정 엣지
graph.add_edge("generate_sql", "execute_sql")
graph.add_edge("format_response", END)
graph.add_edge("direct_answer", END)
graph.add_edge("fail_response", END)

# 조건부 엣지
graph.add_conditional_edges("analyze", route_question)
graph.add_conditional_edges("execute_sql", should_retry)
""")

print("""
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 5단계: 컴파일 및 실행
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

app = graph.compile()  # 실행 가능한 앱으로 컴파일

# 실행
result = app.invoke({"question": "매출 상위 5개 제품은?"})
print(result['final_answer'])
""")

print("=" * 70)
print("이 5단계 패턴이 LangGraph의 전부다.")
print("State 정의 → Node 함수 → 분기 함수 → 그래프 연결 → 컴파일")

---
## 3. 첫 번째 그래프: 질문 분류 → 라우팅 → 응답

질문을 3가지 카테고리(SQL / 일반 / 코드)로 분류하고, 각각 다른 경로로 처리하는 그래프.

```
질문 입력 → [분류] → SQL 질문  → SQL 경로로 처리
                   → 일반 질문 → 일반 경로로 처리
                   → 코드 질문 → 코드 경로로 처리
```

LangGraph 없이 **순수 Python**으로 그래프 로직을 시뮬레이션한다.

In [ ]:
# === 질문 분류 그래프 시뮬레이션 ===

class QuestionRouterGraph:
    """질문을 분류하여 다른 경로로 라우팅하는 그래프 시뮬레이션"""
    
    def __init__(self):
        self.trace = []  # 실행 경로 추적
    
    # --- Node 함수들 ---
    def classify_question(self, state: dict) -> dict:
        """Node: 질문 분류"""
        self.trace.append('classify')
        question = state['question'].lower()
        
        sql_keywords = ['매출', '조회', '테이블', '데이터', '평균', '합계', '상위', 'select']
        code_keywords = ['함수', '코드', '구현', '알고리즘', '클래스', 'python', 'def']
        
        if any(kw in question for kw in sql_keywords):
            state['category'] = 'sql'
        elif any(kw in question for kw in code_keywords):
            state['category'] = 'code'
        else:
            state['category'] = 'general'
        
        print(f"  [classify] '{state['question']}' → 카테고리: {state['category']}")
        return state
    
    def handle_sql(self, state: dict) -> dict:
        """Node: SQL 질문 처리"""
        self.trace.append('sql_handler')
        state['response'] = f"[SQL 경로] '{state['question']}'에 대한 SQL을 생성하고 실행합니다."
        state['sql'] = f"SELECT ... FROM ... -- {state['question']}에 대한 쿼리"
        print(f"  [sql_handler] SQL 생성: {state['sql'][:50]}...")
        return state
    
    def handle_general(self, state: dict) -> dict:
        """Node: 일반 질문 처리"""
        self.trace.append('general_handler')
        state['response'] = f"[일반 경로] '{state['question']}'에 대해 직접 답변합니다."
        print(f"  [general_handler] 직접 응답 생성")
        return state
    
    def handle_code(self, state: dict) -> dict:
        """Node: 코드 질문 처리"""
        self.trace.append('code_handler')
        state['response'] = f"[코드 경로] '{state['question']}'에 대한 코드를 생성합니다."
        print(f"  [code_handler] 코드 생성")
        return state
    
    def format_output(self, state: dict) -> dict:
        """Node: 최종 출력 포맷팅"""
        self.trace.append('format')
        state['final'] = f"최종 응답: {state['response']}"
        print(f"  [format] 최종 응답 생성 완료")
        return state
    
    # --- Conditional Edge ---
    def route(self, state: dict) -> str:
        """Conditional Edge: 카테고리에 따라 라우팅"""
        return {'sql': 'sql_handler', 'general': 'general_handler', 'code': 'code_handler'}[state['category']]
    
    # --- 그래프 실행 ---
    def invoke(self, question: str) -> dict:
        """그래프 실행: START → classify → route → handler → format → END"""
        self.trace = []
        state = {'question': question}
        
        # START → classify
        state = self.classify_question(state)
        
        # classify → conditional edge → handler
        next_node = self.route(state)
        handler = {'sql_handler': self.handle_sql, 
                   'general_handler': self.handle_general, 
                   'code_handler': self.handle_code}[next_node]
        state = handler(state)
        
        # handler → format → END
        state = self.format_output(state)
        
        return state


# === 3개 질문으로 테스트 ===
router = QuestionRouterGraph()

test_questions = [
    "지난 달 매출 상위 5개 제품을 조회해줘",
    "LangGraph가 뭔가요?",
    "Python으로 이진 탐색 함수를 구현해줘",
]

results = []
for q in test_questions:
    print(f"\n{'='*60}")
    print(f"질문: {q}")
    print('-'*60)
    result = router.invoke(q)
    results.append({'question': q, 'category': result['category'], 'trace': router.trace.copy()})
    print(f"실행 경로: {' → '.join(router.trace)}")

# === 경로 시각화 ===
fig, ax = plt.subplots(figsize=(14, 5))

colors_map = {'sql': '#ff6b6b', 'general': '#51cf66', 'code': '#339af0'}
node_labels = ['classify', 'sql_handler', 'general_handler', 'code_handler', 'format']
node_positions = {'classify': (1, 2), 'sql_handler': (3, 3.5), 
                  'general_handler': (3, 2), 'code_handler': (3, 0.5), 'format': (5, 2)}

# 노드 그리기
for name, (x, y) in node_positions.items():
    color = '#ffa94d' if name == 'classify' else colors_map.get(
        name.replace('_handler', ''), '#51cf66')
    box = mpatches.FancyBboxPatch((x-0.6, y-0.3), 1.2, 0.6,
                                  boxstyle="round,pad=0.05",
                                  facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(box)
    display_name = name.replace('_handler', '\nhandler').replace('classify', 'classify\n(분류)')
    ax.text(x, y, display_name, ha='center', va='center', fontsize=9, fontweight='bold', color='white')

# 경로 화살표
path_colors = ['#ff6b6b', '#51cf66', '#339af0']
path_labels = ['SQL 질문', '일반 질문', '코드 질문']

for i, r in enumerate(results):
    handler = r['category'] + '_handler'
    cx, cy = node_positions['classify']
    hx, hy = node_positions[handler]
    fx, fy = node_positions['format']
    
    offset = (i - 1) * 0.08
    ax.annotate('', xy=(hx-0.6, hy+offset), xytext=(cx+0.6, cy+offset),
                arrowprops=dict(arrowstyle='->', lw=2, color=path_colors[i], alpha=0.7))
    ax.annotate('', xy=(fx-0.6, fy+offset), xytext=(hx+0.6, hy+offset),
                arrowprops=dict(arrowstyle='->', lw=2, color=path_colors[i], alpha=0.7))

# 범례
for i, label in enumerate(path_labels):
    ax.plot([], [], color=path_colors[i], linewidth=2, label=label)
ax.legend(loc='upper right', fontsize=10)

ax.set_xlim(0, 6.5)
ax.set_ylim(-0.2, 4.5)
ax.set_title('질문 분류 라우팅 그래프 — 3가지 경로', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

print("\nConditional Edge의 핵심: 같은 그래프, 다른 경로.")
print("질문의 종류에 따라 다른 노드를 방문한다.")

---
## 4. Human-in-the-Loop: 사람 검증 지점

위험한 작업(DELETE, DROP, UPDATE)은 자동 실행하면 안 된다.  
LangGraph는 `interrupt_before` / `interrupt_after`로 **사람의 승인을 기다리는 지점**을 설정할 수 있다.

| 패턴 | 적용 시점 | 이유 |
|------|----------|------|
| `interrupt_before` | 실행 전 승인 | DELETE/UPDATE 등 위험 SQL |
| `interrupt_after` | 실행 후 확인 | 결과 검증이 필요한 경우 |
| 자동 진행 | 승인 불필요 | SELECT 등 읽기 전용 쿼리 |

```python
# LangGraph에서의 HITL 설정
app = graph.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["execute_sql"]  # 실행 전 중단
)

# 사용자 승인 후 재개
result = app.invoke(None, config)  # 중단된 지점부터 재개
```

In [ ]:
# === Human-in-the-Loop 시뮬레이션 ===

class HITLSimulator:
    """위험한 SQL은 사람 승인이 필요한 흐름 시뮬레이션"""
    
    DANGEROUS_KEYWORDS = ['DELETE', 'DROP', 'UPDATE', 'TRUNCATE', 'ALTER']
    
    def __init__(self):
        self.trace = []
    
    def generate_sql(self, state: dict) -> dict:
        """Node: SQL 생성"""
        self.trace.append(('generate_sql', 'normal'))
        print(f"  [generate_sql] SQL: {state['sql']}")
        return state
    
    def check_danger(self, state: dict) -> str:
        """Conditional Edge: 위험도 판단"""
        sql_upper = state['sql'].upper()
        is_dangerous = any(kw in sql_upper for kw in self.DANGEROUS_KEYWORDS)
        
        if is_dangerous:
            print(f"  [check_danger] 위험 SQL 감지! → 사람 승인 필요")
            return 'human_approval'
        else:
            print(f"  [check_danger] 안전한 SQL → 자동 실행")
            return 'execute'
    
    def human_approval(self, state: dict, approve: bool) -> dict:
        """Node: 사람 승인 (interrupt point)"""
        self.trace.append(('human_approval', 'approve' if approve else 'reject'))
        if approve:
            state['approved'] = True
            print(f"  [human_approval] 승인됨 → 실행 진행")
        else:
            state['approved'] = False
            print(f"  [human_approval] 거부됨 → 작업 취소")
        return state
    
    def execute_sql(self, state: dict) -> dict:
        """Node: SQL 실행"""
        self.trace.append(('execute_sql', 'success'))
        state['result'] = f"실행 완료: {state['sql']}"
        print(f"  [execute_sql] 실행 성공")
        return state
    
    def cancel(self, state: dict) -> dict:
        """Node: 작업 취소"""
        self.trace.append(('cancel', 'cancelled'))
        state['result'] = "작업이 취소되었습니다."
        print(f"  [cancel] 작업 취소됨")
        return state
    
    def invoke(self, sql: str, human_decision: bool = True) -> dict:
        """그래프 실행"""
        self.trace = []
        state = {'sql': sql}
        
        # generate_sql
        state = self.generate_sql(state)
        
        # check_danger (conditional edge)
        route = self.check_danger(state)
        
        if route == 'human_approval':
            # INTERRUPT: 사람 승인 대기
            state = self.human_approval(state, human_decision)
            if state['approved']:
                state = self.execute_sql(state)
            else:
                state = self.cancel(state)
        else:
            # 안전한 SQL → 자동 실행
            state = self.execute_sql(state)
        
        return state


# === 4가지 시나리오 테스트 ===
hitl = HITLSimulator()

scenarios = [
    ("SELECT * FROM products LIMIT 10", True, "안전한 SELECT"),
    ("DELETE FROM users WHERE id = 42", True, "위험한 DELETE → 승인"),
    ("DROP TABLE temp_data", False, "위험한 DROP → 거부"),
    ("UPDATE orders SET status = 'cancelled'", True, "위험한 UPDATE → 승인"),
]

scenario_results = []
for sql, decision, desc in scenarios:
    print(f"\n{'='*60}")
    print(f"시나리오: {desc}")
    print(f"SQL: {sql}")
    print('-'*60)
    result = hitl.invoke(sql, decision)
    scenario_results.append({
        'desc': desc, 'sql': sql, 'trace': hitl.trace.copy(), 
        'result': result.get('result', '')
    })
    print(f"결과: {result.get('result', '')}")

# === 경로 시각화 ===
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (ax, sr) in enumerate(zip(axes.flat, scenario_results)):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 8)
    ax.axis('off')
    ax.set_title(sr['desc'], fontsize=11, fontweight='bold')
    
    # 노드 위치
    nodes = [
        (2, 7, 'SQL 생성', '#339af0'),
        (5, 7, '위험도\n판단', '#ffa94d'),
    ]
    
    # 경로에 따라 다른 노드
    has_approval = any(t[0] == 'human_approval' for t in sr['trace'])
    was_approved = any(t[0] == 'human_approval' and t[1] == 'approve' for t in sr['trace'])
    was_cancelled = any(t[0] == 'cancel' for t in sr['trace'])
    
    if has_approval:
        nodes.append((5, 4.5, '사람 승인\nINTERRUPT', '#ff6b6b'))
        if was_approved:
            nodes.append((5, 2, 'SQL 실행', '#51cf66'))
        else:
            nodes.append((5, 2, '작업 취소', '#c4c4c4'))
    else:
        nodes.append((8, 5, '자동 실행', '#51cf66'))
    
    # 노드 그리기
    for nx, ny, label, color in nodes:
        box = mpatches.FancyBboxPatch((nx-0.9, ny-0.45), 1.8, 0.9,
                                      boxstyle="round,pad=0.05",
                                      facecolor=color, edgecolor='black', linewidth=2)
        ax.add_patch(box)
        ax.text(nx, ny, label, ha='center', va='center', fontsize=8, 
                fontweight='bold', color='white')
    
    # 화살표
    ax.annotate('', xy=(4.1, 7), xytext=(2.9, 7),
                arrowprops=dict(arrowstyle='->', lw=2, color='#333'))
    
    if has_approval:
        ax.annotate('', xy=(5, 5.0), xytext=(5, 6.5),
                    arrowprops=dict(arrowstyle='->', lw=2, color='red'))
        ax.text(5.5, 5.8, '위험!', fontsize=8, color='red', fontweight='bold')
        ax.annotate('', xy=(5, 2.5), xytext=(5, 4.0),
                    arrowprops=dict(arrowstyle='->', lw=2, 
                                   color='green' if was_approved else '#999'))
        label_text = '승인' if was_approved else '거부'
        label_color = 'green' if was_approved else '#999'
        ax.text(5.5, 3.3, label_text, fontsize=8, color=label_color, fontweight='bold')
    else:
        ax.annotate('', xy=(7.1, 5.3), xytext=(5.9, 6.5),
                    arrowprops=dict(arrowstyle='->', lw=2, color='green'))
        ax.text(7, 6.2, '안전', fontsize=8, color='green', fontweight='bold')

plt.suptitle('Human-in-the-Loop: 위험 SQL은 사람이 판단한다', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nHITL 핵심: 자동화와 안전성의 균형.")
print("SELECT → 자동 실행, DELETE/DROP/UPDATE → 사람 승인 후 실행.")

---
## 5. 멀티스텝 SQL 에이전트 (핵심 실습)

실패 시 자동으로 SQL을 재생성하는 **자기 교정(Self-Correcting)** 에이전트.

```
질문 분석 → SQL 생성 → SQL 실행 → 성공? → 응답 생성
                         ↑         ↓
                         └── 실패 (재시도 < 3)
                                   ↓
                              재시도 >= 3 → 실패 응답
```

**State:**
- `question`: 사용자 질문
- `sql`: 생성된 SQL
- `result`: 실행 결과
- `error`: 에러 메시지
- `retry_count`: 재시도 횟수

**Nodes:** analyze_question, generate_sql, execute_sql, validate_result, format_response  
**Edges:** execute → 성공? → format / 실패? → retry(max 3) → generate_sql

In [ ]:
# === 멀티스텝 SQL 에이전트 구현 (순수 Python) ===

class SQLGraphAgent:
    """LangGraph 스타일 멀티스텝 SQL 에이전트 시뮬레이션
    
    State: question, sql, result, error, retry_count
    Nodes: analyze_question, generate_sql, execute_sql, validate_result, format_response
    Edges: execute→성공→format / 실패→retry(max 3)→generate_sql
    """
    
    MAX_RETRIES = 3
    
    # --- 시뮬레이션용 SQL 패턴 ---
    SQL_PATTERNS = {
        '매출': 'SELECT product, SUM(amount) FROM sales GROUP BY product ORDER BY SUM(amount) DESC LIMIT 5',
        '사용자': 'SELECT name, email FROM users WHERE active = 1',
        '주문': 'SELECT o.id, o.total FROM orders o JOIN customers c ON o.cust_id = c.id',
    }
    
    # --- 시뮬레이션용 에러 시나리오 ---
    ERROR_SCENARIOS = {
        'error_test': [
            ('SELECT * FORM users', 'Syntax error: FORM → FROM'),  # 1차 시도: 문법 오류
            ('SELECT * FROM userss', 'Table not found: userss'),    # 2차 시도: 테이블명 오류
            ('SELECT * FROM users', None),                          # 3차 시도: 성공
        ],
        'complex_test': [
            ('SELECT a, b FROM x JOIN y', 'Ambiguous column: a'),
            ('SELECT x.a, y.b FROM x JOIN y ON x.id = y.xid', None),
        ],
    }
    
    def __init__(self):
        self.trace = []  # [(node_name, status)] 실행 추적
    
    # === Node 함수들 ===
    
    def analyze_question(self, state: dict) -> dict:
        """Node 1: 질문 분석 — 의도 파악 + 테이블 식별"""
        self.trace.append(('analyze_question', 'processing'))
        question = state['question']
        
        # 키워드 기반 테이블 식별 (시뮬레이션)
        tables = []
        if '매출' in question or '제품' in question:
            tables = ['sales', 'products']
        elif '사용자' in question or '회원' in question:
            tables = ['users']
        elif '주문' in question:
            tables = ['orders', 'customers']
        else:
            tables = ['unknown']
        
        state['tables'] = tables
        state['retry_count'] = 0
        state['error'] = ''
        
        print(f"  [analyze_question] 질문: '{question}'")
        print(f"  [analyze_question] 관련 테이블: {tables}")
        return state
    
    def generate_sql(self, state: dict) -> dict:
        """Node 2: SQL 생성 — 에러 있으면 수정 반영"""
        self.trace.append(('generate_sql', f"attempt_{state['retry_count']+1}"))
        
        scenario = state.get('scenario', 'normal')
        
        if scenario in self.ERROR_SCENARIOS:
            attempts = self.ERROR_SCENARIOS[scenario]
            idx = min(state['retry_count'], len(attempts) - 1)
            state['sql'] = attempts[idx][0]
        else:
            # 키워드 매칭으로 SQL 생성
            sql = 'SELECT * FROM data'
            for keyword, pattern in self.SQL_PATTERNS.items():
                if keyword in state['question']:
                    sql = pattern
                    break
            state['sql'] = sql
        
        state['retry_count'] += 1
        
        prefix = "[재생성]" if state['retry_count'] > 1 else "[생성]"
        print(f"  [generate_sql] {prefix} (시도 {state['retry_count']}/{self.MAX_RETRIES}): {state['sql']}")
        
        if state.get('error'):
            print(f"  [generate_sql] 이전 에러 반영: {state['error']}")
        
        return state
    
    def execute_sql(self, state: dict) -> dict:
        """Node 3: SQL 실행"""
        scenario = state.get('scenario', 'normal')
        
        if scenario in self.ERROR_SCENARIOS:
            attempts = self.ERROR_SCENARIOS[scenario]
            idx = min(state['retry_count'] - 1, len(attempts) - 1)
            error = attempts[idx][1]
            
            if error:
                state['error'] = error
                state['result'] = ''
                self.trace.append(('execute_sql', 'error'))
                print(f"  [execute_sql] 에러 발생: {error}")
            else:
                state['error'] = ''
                state['result'] = '[(product_a, 5000), (product_b, 3000)]'
                self.trace.append(('execute_sql', 'success'))
                print(f"  [execute_sql] 실행 성공! 결과: {state['result']}")
        else:
            state['error'] = ''
            state['result'] = '[(product_a, 15000), (product_b, 12000), (product_c, 9800)]'
            self.trace.append(('execute_sql', 'success'))
            print(f"  [execute_sql] 실행 성공! 결과: {state['result'][:60]}...")
        
        return state
    
    def validate_result(self, state: dict) -> str:
        """Conditional Edge: 결과 검증 → 다음 경로 결정"""
        if not state.get('error'):
            self.trace.append(('validate_result', 'pass'))
            print(f"  [validate_result] 검증 통과 → format_response")
            return 'format_response'
        elif state['retry_count'] < self.MAX_RETRIES:
            self.trace.append(('validate_result', 'retry'))
            print(f"  [validate_result] 실패 → 재시도 ({state['retry_count']}/{self.MAX_RETRIES})")
            return 'generate_sql'
        else:
            self.trace.append(('validate_result', 'max_retry'))
            print(f"  [validate_result] 최대 재시도 초과 → fail_response")
            return 'fail_response'
    
    def format_response(self, state: dict) -> dict:
        """Node 4: 성공 응답 생성"""
        self.trace.append(('format_response', 'done'))
        state['final_answer'] = f"질문: {state['question']}\nSQL: {state['sql']}\n결과: {state['result']}"
        print(f"  [format_response] 최종 응답 생성 완료")
        return state
    
    def fail_response(self, state: dict) -> dict:
        """Node 5: 실패 응답"""
        self.trace.append(('fail_response', 'failed'))
        state['final_answer'] = f"죄송합니다. '{state['question']}'을 처리할 수 없습니다. 마지막 에러: {state['error']}"
        print(f"  [fail_response] 실패 응답 생성")
        return state
    
    # === 그래프 실행 엔진 ===
    
    def invoke(self, question: str, scenario: str = 'normal') -> dict:
        """그래프 실행: analyze → generate → execute → validate → (format | retry | fail)"""
        self.trace = []
        state = {'question': question, 'scenario': scenario}
        
        # START → analyze_question
        state = self.analyze_question(state)
        
        # analyze → generate_sql
        state = self.generate_sql(state)
        
        # 루프: execute → validate → (format | retry | fail)
        while True:
            state = self.execute_sql(state)
            next_node = self.validate_result(state)
            
            if next_node == 'format_response':
                state = self.format_response(state)
                break
            elif next_node == 'fail_response':
                state = self.fail_response(state)
                break
            else:  # retry → generate_sql
                state = self.generate_sql(state)
        
        return state


# === 3개 테스트 케이스 실행 ===
agent = SQLGraphAgent()
test_cases = [
    ("매출 상위 5개 제품을 조회해줘", 'normal', "정상 실행"),
    ("사용자 목록을 보여줘", 'error_test', "SQL 에러 → 재시도 → 성공"),
    ("주문과 고객을 조인해서 보여줘", 'complex_test', "복잡 쿼리 → 1회 재시도 → 성공"),
]

all_traces = []
for question, scenario, desc in test_cases:
    print(f"\n{'='*70}")
    print(f"테스트: {desc}")
    print(f"질문: {question}")
    print('-'*70)
    result = agent.invoke(question, scenario)
    all_traces.append({'desc': desc, 'trace': agent.trace.copy(), 'question': question})
    print(f"\n최종 응답:\n{result.get('final_answer', 'N/A')}")
    print(f"실행 경로: {' → '.join([t[0] for t in agent.trace])}")
    print(f"총 노드 방문: {len(agent.trace)}회")

In [ ]:
# === 에이전트 실행 흐름 시각화 ===

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# 노드별 위치 정의
node_positions = {
    'analyze_question': (0, 4),
    'generate_sql': (1, 4),
    'execute_sql': (2, 4),
    'validate_result': (3, 4),
    'format_response': (4, 5),
    'fail_response': (4, 3),
}

node_colors = {
    'analyze_question': '#339af0',
    'generate_sql': '#ffa94d',
    'execute_sql': '#ff6b6b',
    'validate_result': '#ffa94d',
    'format_response': '#51cf66',
    'fail_response': '#c4c4c4',
}

node_short = {
    'analyze_question': 'analyze',
    'generate_sql': 'generate',
    'execute_sql': 'execute',
    'validate_result': 'validate',
    'format_response': 'format',
    'fail_response': 'fail',
}

for idx, (ax, trace_info) in enumerate(zip(axes, all_traces)):
    ax.set_xlim(-0.8, 5.5)
    ax.set_ylim(1.5, 6.5)
    ax.axis('off')
    ax.set_title(trace_info['desc'], fontsize=11, fontweight='bold')
    
    # 모든 노드 그리기 (방문하지 않은 노드는 투명하게)
    visited_nodes = set(t[0] for t in trace_info['trace'])
    
    for name, (x, y) in node_positions.items():
        alpha = 1.0 if name in visited_nodes else 0.2
        color = node_colors[name]
        
        box = mpatches.FancyBboxPatch((x-0.4, y-0.3), 0.8, 0.6,
                                      boxstyle="round,pad=0.05",
                                      facecolor=color, edgecolor='black', 
                                      linewidth=2, alpha=alpha)
        ax.add_patch(box)
        ax.text(x, y, node_short[name], ha='center', va='center', 
                fontsize=8, fontweight='bold', color='white', alpha=alpha)
    
    # 실행 경로 화살표
    trace_nodes = [t[0] for t in trace_info['trace']]
    
    for i in range(len(trace_nodes) - 1):
        src = trace_nodes[i]
        dst = trace_nodes[i + 1]
        
        if src in node_positions and dst in node_positions:
            sx, sy = node_positions[src]
            dx, dy = node_positions[dst]
            
            # 재시도 루프: validate → generate_sql
            if src == 'validate_result' and dst == 'generate_sql':
                ax.annotate('', xy=(dx+0.2, dy+0.35), xytext=(sx-0.2, sy+0.35),
                            arrowprops=dict(arrowstyle='->', lw=2, color='red',
                                           connectionstyle='arc3,rad=0.5'))
            else:
                ax.annotate('', xy=(dx-0.4, dy), xytext=(sx+0.4, sy),
                            arrowprops=dict(arrowstyle='->', lw=1.5, color='#333', alpha=0.7))
    
    # 방문 횟수 표시
    visit_counts = {}
    for t in trace_nodes:
        visit_counts[t] = visit_counts.get(t, 0) + 1
    
    for name, count in visit_counts.items():
        if count > 1 and name in node_positions:
            x, y = node_positions[name]
            ax.text(x+0.35, y+0.35, f'x{count}', fontsize=8, color='red', 
                    fontweight='bold', ha='center',
                    bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='red'))
    
    # 총 스텝
    ax.text(2.5, 1.8, f'총 {len(trace_nodes)} 스텝', ha='center', fontsize=10, 
            fontweight='bold', color='#333',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#f0f0f0', edgecolor='#999'))

plt.suptitle('멀티스텝 SQL 에이전트 — 실행 흐름 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n핵심 관찰:")
print("  테스트 1: 정상 → 최단 경로 (5 스텝)")
print("  테스트 2: 2회 실패 후 성공 → 재시도 루프 발생")
print("  테스트 3: 1회 실패 후 성공 → 자기 교정 능력")
print("\n이것이 Graph의 힘이다: 실패해도 루프를 돌아 재시도할 수 있다.")

---
## 6. 상태 관리와 체크포인팅

체크포인팅은 그래프의 **실행 상태를 저장하고 복원**하는 기능이다.

| 기능 | 설명 | 용도 |
|------|------|------|
| **MemorySaver** | 메모리 기반 체크포인터 | 개발/테스트 |
| **SqliteSaver** | SQLite 기반 체크포인터 | 프로덕션 |
| **thread_id** | 대화 식별자 | 멀티 세션 관리 |
| **중단/재개** | interrupt 후 상태 복원 | HITL 패턴 |

체크포인팅이 없으면 HITL은 불가능하다.  
사람이 승인하기 전까지 **그래프의 모든 상태**를 보존해야 하기 때문이다.

```python
# 체크포인터 설정
from langgraph.checkpoint.memory import MemorySaver
checkpointer = MemorySaver()

# thread_id로 대화별 상태 관리
config = {"configurable": {"thread_id": "user_123_session_1"}}

# 동일 thread_id → 이전 상태 이어서 진행
result_1 = app.invoke({"question": "매출 보여줘"}, config)
result_2 = app.invoke({"question": "좀 더 자세히"}, config)  # 이전 상태 기억
```

In [ ]:
# === 체크포인팅 개념 시각화 + 코드 패턴 ===

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 좌측: 체크포인팅 없이 ---
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('체크포인팅 없음 — HITL 불가', fontsize=12, fontweight='bold', color='red')

steps_no_cp = [
    (5, 9, 'SQL 생성', '#339af0'),
    (5, 7, 'INTERRUPT\n(사람 대기)', '#ff6b6b'),
    (5, 5, '상태 소멸!', '#c4c4c4'),
    (5, 3, '처음부터\n다시 시작', '#c4c4c4'),
]

for x, y, label, color in steps_no_cp:
    box = mpatches.FancyBboxPatch((x-1.2, y-0.5), 2.4, 1.0,
                                  boxstyle="round,pad=0.1",
                                  facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center', fontsize=10, fontweight='bold', color='white')

for i in range(len(steps_no_cp)-1):
    ax.annotate('', xy=(5, steps_no_cp[i+1][1]+0.6), xytext=(5, steps_no_cp[i][1]-0.6),
                arrowprops=dict(arrowstyle='->', lw=2, color='#999'))

ax.text(5, 1.5, '문제: 사람이 돌아왔을 때 상태가 없다', 
        ha='center', fontsize=10, color='red', style='italic')

# --- 우측: 체크포인팅 있음 ---
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('체크포인팅 있음 — HITL 가능', fontsize=12, fontweight='bold', color='green')

steps_cp = [
    (5, 9, 'SQL 생성', '#339af0'),
    (5, 7, 'INTERRUPT\n+ 상태 저장', '#ff6b6b'),
    (5, 5, '... 시간 경과 ...', '#ffa94d'),
    (5, 3, '상태 복원\n+ 재개', '#51cf66'),
    (5, 1, 'SQL 실행', '#51cf66'),
]

for x, y, label, color in steps_cp:
    box = mpatches.FancyBboxPatch((x-1.2, y-0.5), 2.4, 1.0,
                                  boxstyle="round,pad=0.1",
                                  facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center', fontsize=10, fontweight='bold', color='white')

for i in range(len(steps_cp)-1):
    ax.annotate('', xy=(5, steps_cp[i+1][1]+0.6), xytext=(5, steps_cp[i][1]-0.6),
                arrowprops=dict(arrowstyle='->', lw=2, color='green'))

# 체크포인트 저장소 표시
storage = mpatches.FancyBboxPatch((7.5, 6.5), 2.0, 1.0,
                                  boxstyle="round,pad=0.1",
                                  facecolor='#eee', edgecolor='#999', linewidth=1.5, linestyle='--')
ax.add_patch(storage)
ax.text(8.5, 7, 'MemorySaver\n(상태 보관)', ha='center', va='center', fontsize=8, color='#666')

ax.annotate('', xy=(7.5, 7), xytext=(6.2, 7),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#999', linestyle='dashed'))
ax.annotate('', xy=(6.2, 3.5), xytext=(7.5, 6.5),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#999', linestyle='dashed'))
ax.text(7.8, 5, '저장/복원', fontsize=8, color='#999', rotation=-70)

plt.suptitle('Checkpointing: HITL의 필수 기반', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# === 코드 패턴 출력 ===
print("\n" + "=" * 70)
print("체크포인팅 코드 패턴")
print("=" * 70)

print("""
from langgraph.checkpoint.memory import MemorySaver

# 1. 체크포인터 생성
checkpointer = MemorySaver()

# 2. 그래프 컴파일 시 체크포인터 연결
app = graph.compile(
    checkpointer=checkpointer,
    interrupt_before=["execute_sql"]  # HITL 중단점
)

# 3. thread_id로 대화별 상태 관리
config = {"configurable": {"thread_id": "user_123_session_1"}}

# 4. 첫 실행 → interrupt에서 중단
result = app.invoke({"question": "DELETE FROM users WHERE id=1"}, config)
# → execute_sql 전에 중단됨

# 5. 사용자 승인 후 재개 (같은 config로)
result = app.invoke(None, config)  # None = 추가 입력 없이 재개
# → 중단된 execute_sql부터 이어서 실행

# 6. 다른 thread_id = 다른 대화
config2 = {"configurable": {"thread_id": "user_456_session_1"}}
# → 완전히 독립된 상태
""")

print("핵심: thread_id가 같으면 이전 상태를 기억한다.")
print("      thread_id가 다르면 완전히 독립된 대화이다.")

---
## 7. 핵심 정리

| 개념 | 핵심 |
|------|------|
| **LangChain vs LangGraph** | Chain=직선도로, Graph=도시 도로망 |
| **StateGraph** | 상태를 공유하는 노드-엣지 그래프 |
| **Conditional Edges** | 결과에 따라 다른 경로로 분기 |
| **Human-in-the-Loop** | 위험 작업 전 사람 승인 (interrupt) |
| **멀티스텝 에이전트** | 실패→재시도 루프로 자기 교정 |
| **Checkpointing** | 상태 저장/복원, HITL의 필수 기반 |

### 핵심 원칙

> **Chain은 직선도로, Graph는 도시 전체 도로망이다.**  
> 현실의 문제는 "질문→답변"의 직선이 아니라, 분기하고 실패하고 재시도하는 복잡한 경로를 거친다.  
> LangGraph는 이 복잡성을 구조적으로 다룬다.

### 다음: 03_Text2SQL_에이전트_구축.ipynb

이번에 배운 LangGraph 구조를 실제 Text-to-SQL 에이전트에 적용한다.

In [ ]:
# === Phase 10-2 체크포인트 ===

print("=" * 70)
print("Phase 10-2 체크포인트: LangGraph 마스터하기")
print("=" * 70)

checkpoints = [
    ("LangGraph와 LangChain의 근본적 차이를 설명할 수 있는가?",
     "Chain=직선(A→B→C), Graph=분기+루프. 현실 문제는 직선이 아니다."),
    
    ("StateGraph의 Node, Edge, Conditional Edge 개념을 이해하는가?",
     "Node=처리단계, Edge=고정연결, Conditional Edge=조건분기."),
    
    ("실패 시 자동 재시도하는 멀티스텝 에이전트를 설계할 수 있는가?",
     "execute→validate→(성공→format | 실패→retry<3→generate | 초과→fail)"),
    
    ("Human-in-the-Loop 패턴의 필요성과 구현 방법을 설명할 수 있는가?",
     "interrupt_before로 위험 SQL 전에 중단, 승인 후 재개."),
    
    ("Checkpointing을 활용하여 에이전트 상태를 저장/복원할 수 있는가?",
     "MemorySaver + thread_id로 대화별 상태 관리. HITL의 필수 기반."),
]

for i, (question, answer) in enumerate(checkpoints, 1):
    print(f"\n{'─'*60}")
    print(f"  [{i}] {question}")
    print(f"      → {answer}")

print(f"\n{'='*70}")
print("모든 체크포인트 확인 완료.")
print("\n핵심 원칙:")
print('  "Chain은 직선도로, Graph는 도시 전체 도로망이다."')
print('  "현실의 문제는 직선이 아니라 분기와 루프로 이루어져 있다."')